# Ribosome-Cascade - Scale Experiment (chunks=64, hidden=768, 100K steps on C4)

**Hypothesis under test:** the 0-0.3% LAMBADA dead zone we've been stuck in is a
*compression-ratio + scale* problem, not an architectural one. At chunks=16 with
~36M params and ~100M tokens we see PPL gain but zero semantic recovery. At
chunks=64 (compression ratio 4) with ~105M params and ~1.6B tokens of C4 we land
far enough up the loss curve to distinguish.

**Decision rule (pre-registered):**
- LAMBADA acc > 10%  -> architecture is sound, just undertrained at small scale. Pursue.
- LAMBADA acc 1-10%  -> marginal. Compare to matched-FLOPs baseline before deciding.
- LAMBADA acc < 1%   -> hourglass as specified cannot do last-word prediction.
                        Reframe as compression research (PPL/bit vs ratio Pareto).

**What makes this run different from the cluster sweep:**
1. Lower compression ratio (4, not 16) - this is the architectural lever we haven't moved.
2. A100 40GB with bf16 - 4-5x more tokens/sec than our local 3060 Ti / 4070 Ti.
3. C4 streaming (not wikitext-103) - training distribution is orders of magnitude larger.
4. 100K steps * batch 64 * seq 256 ~= 1.6B tokens seen - roughly GPT-2 Small's budget.
5. Drive checkpointing every 2K steps - fully resumable across Colab session drops.


In [ ]:
# Single setup cell. Run this first. Everything downstream depends on CONFIG.
import subprocess, os, json, time, math, sys, random

# ---- deps: Colab ships transformers/datasets/accelerate preinstalled.
#      Only install if actually missing - avoids numpy ABI breakage (v1/v2).
_NEED = []
for _pkg in ('transformers', 'datasets', 'accelerate'):
    try: __import__(_pkg)
    except ImportError: _NEED.append(_pkg)
if _NEED:
    print('installing:', _NEED)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + _NEED)
else:
    import transformers as _t, datasets as _d, accelerate as _a
    print(f'preinstalled: transformers {_t.__version__}, datasets {_d.__version__}, '
          f'accelerate {_a.__version__}')

# ---- canonical config (change here, nowhere else) ----
CONFIG = dict(
    # model
    vocab_size=50257, hidden_size=768, n_heads=12,
    embed_layers=3, upper_layers=3, reverse_layers=2,
    max_seq_len=256, n_chunks=64,
    # training
    total_steps=100_000, warmup_steps=5_000, alpha_ramp_steps=10_000,
    max_lr=3e-4, min_lr=3e-5,
    batch_size=64, grad_accum=1,
    weight_decay=0.1, beta1=0.9, beta2=0.95, grad_clip=1.0,
    dtype="bf16",
    # data
    train_dataset="c4", train_subset="en",
    # logging / eval
    log_every=100, eval_every=2_000, ckpt_every=2_000,
    mid_eval_wt103_batches=50, mid_eval_c4_batches=50,
    seed=42,
)
print('CONFIG defined.')
print(json.dumps(CONFIG, indent=2))

# ---- GPU check ----
try:
    print(subprocess.check_output(['nvidia-smi',
        '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader']).decode())
except Exception as e:
    print('nvidia-smi failed:', e)

# ---- Drive mount + paths ----
from google.colab import drive
drive.mount('/content/drive')
RUN_TAG    = "ribo64c_h768_100k_c4"
DRIVE_BASE = "/content/drive/MyDrive/Ribosome-Cascade/scale_experiment_corrected"
RUN_DIR    = os.path.join(DRIVE_BASE, RUN_TAG)
CKPT_PATH  = os.path.join(RUN_DIR, "checkpoint.pt")
BEST_PATH  = os.path.join(RUN_DIR, "best.pt")
RESULTS    = os.path.join(RUN_DIR, "results.json")
LOG_PATH   = os.path.join(RUN_DIR, "train.log")
os.makedirs(RUN_DIR, exist_ok=True)
print('Output dir:', RUN_DIR)
print('Resume from:', CKPT_PATH, 'exists:', os.path.exists(CKPT_PATH))


In [ ]:
%%writefile /content/native_arch_v1.py
# Ribosome-Cascade Track 2: Native Architecture
# ================================================
# Architecture Specification
#
# A transformer trained from scratch where the ribosome is a
# first-class architectural component, not a bolt-on.
#
# ARCHITECTURE OVERVIEW
# =====================
#
#   Input tokens
#       │
#       ▼
#   [Token Embedding + Positional Encoding]
#       │
#       ▼
#   ┌─────────────────────────────┐
#   │  LOWER TRANSFORMER (4 layers)  │  ← Standard token-level processing
#   │  Token-level self-attention     │  ← Builds contextual representations
#   └─────────────────────────────┘
#       │
#       ▼  (seq_len tokens, each with hidden_size features)
#   ┌─────────────────────────────┐
#   │  RIBOSOME LAYER              │  ← Scores, groups, compresses
#   │  1. Score importance per token│
#   │  2. Soft boundary detection   │
#   │  3. Chunk encoding (Perceiver)│
#   └─────────────────────────────┘
#       │
#       ▼  (n_chunks metatokens, each with hidden_size features)
#   ┌─────────────────────────────┐
#   │  CASCADE PROCESSOR (2 layers) │  ← Priority-ordered chunk processing
#   │  Causal attention by weight   │  ← Heaviest chunk = anchor
#   └─────────────────────────────┘
#       │
#       ▼
#   ┌─────────────────────────────┐
#   │  UPPER TRANSFORMER (4 layers) │  ← Chunk-level processing
#   │  Standard self-attention      │  ← But on metatokens, not tokens
#   └─────────────────────────────┘
#       │
#       ▼
#   ┌─────────────────────────────┐
#   │  CHUNK DECODER               │  ← Expand chunks back to tokens
#   │  Cross-attention: tokens      │
#   │  attend to processed chunks   │
#   └─────────────────────────────┘
#       │
#       ▼
#   [LM Head → vocab logits]
#
#
# PARAMETER BUDGET (~250M on RTX 5090 32GB)
# ==========================================
#   hidden_size = 768
#   vocab_size = 50257 (GPT-2 tokenizer)
#   max_seq_len = 1024
#   n_chunks = variable (learned boundaries, target ~seq_len/4)
#
#   Token embedding:     768 * 50257           = ~38.6M
#   Lower transformer:   4 layers * ~7M each   = ~28M
#   Ribosome layer:      scorer + boundary      = ~2M
#   Chunk encoder:       Perceiver cross-attn   = ~5M
#   Cascade processor:   2 layers               = ~14M
#   Upper transformer:   4 layers               = ~28M
#   Chunk decoder:       cross-attn + expand    = ~5M
#   LM head:             tied with embedding    = 0 (weight tying)
#   ─────────────────────────────────────────
#   Total:                                      ≈ 121M
#
#   Room to grow: could double layer counts or hidden_size
#
#
# KEY DESIGN DECISIONS
# ====================
#
# 1. DIFFERENTIABLE GROUPING: Gumbel-softmax boundary prediction
#    - Ribosome outputs boundary probability per token position
#    - During training: Gumbel-softmax samples soft boundaries
#    - During inference: hard argmax boundaries
#    - This avoids the two-phase training complexity
#
# 2. VARIABLE-LENGTH CHUNKS: Perceiver-style chunk encoder
#    - Predicts K boundary positions (K = target number of chunks)
#    - Tokens between boundaries form a chunk
#    - Each chunk compressed via cross-attention with a learnable query
#    - NOT mean-pooling — learned compression
#
# 3. PRIORITY CASCADE: causal attention by chunk weight
#    - Each chunk's weight = sum of token importance scores
#    - Chunks sorted by weight descending
#    - Causal mask: chunk i attends only to chunks 0..i-1
#    - Heaviest chunk (anchor) processed first, establishes context
#    - Lighter chunks conditioned on anchor
#
# 4. TRAINING DATA: OpenWebText (subset)
#    - wikitext-2 too small for from-scratch training
#    - OpenWebText ~38GB, use 1-2GB subset initially
#    - Can scale up as architecture validates
#
# 5. TWO-STAGE TRAINING:
#    - Stage 1 (warmup): Train without ribosome layer active
#      Lower + Upper transformers learn basic LM, ribosome scores
#      accumulate but don't affect forward pass. 5-10% of total steps.
#    - Stage 2 (full): Enable ribosome compression. The model must
#      now route through chunks. Gradual activation via alpha ramp.

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# BUILDING BLOCKS
# ============================================================

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    
    def forward(self, x):
        norm = x.float().pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (x.float() * norm).type_as(x) * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        t = torch.arange(max_seq_len).float()
        freqs = torch.outer(t, inv_freq)
        self.register_buffer("cos_cached", freqs.cos())
        self.register_buffer("sin_cached", freqs.sin())
    
    def forward(self, seq_len):
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]


def apply_rotary(x, cos, sin):
    """Apply rotary embeddings. x: (B, n_heads, S, head_dim)"""
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]
    cos = cos[:x.shape[2]].unsqueeze(0).unsqueeze(0)
    sin = sin[:x.shape[2]].unsqueeze(0).unsqueeze(0)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)


class SelfAttention(nn.Module):
    def __init__(self, hidden_size, n_heads, rope=None):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = hidden_size // n_heads
        self.qkv = nn.Linear(hidden_size, 3 * hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.rope = rope
    
    def forward(self, x, causal_mask=None):
        B, S, H = x.shape
        qkv = self.qkv(x).view(B, S, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)  # each: (B, S, n_heads, head_dim)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        
        if self.rope is not None:
            cos, sin = self.rope(S)
            q = apply_rotary(q, cos, sin)
            k = apply_rotary(k, cos, sin)
        
        scale = math.sqrt(self.head_dim)
        attn = torch.matmul(q, k.transpose(-2, -1)) / scale
        
        if causal_mask is not None:
            attn = attn.masked_fill(causal_mask, float('-inf'))
        
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, S, H)
        return self.o_proj(out)


class FFN(nn.Module):
    """SwiGLU feed-forward."""
    def __init__(self, hidden_size, ff_mult=4):
        super().__init__()
        ff_dim = int(hidden_size * ff_mult * 2 / 3)  # SwiGLU correction
        self.w1 = nn.Linear(hidden_size, ff_dim, bias=False)
        self.w2 = nn.Linear(ff_dim, hidden_size, bias=False)
        self.w3 = nn.Linear(hidden_size, ff_dim, bias=False)
    
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


def sinusoidal_position_encoding(positions, dim):
    """Compute sinusoidal positional encoding at arbitrary (fractional) positions.
    
    Args:
        positions: (B, K) — fractional positions in sequence
        dim: hidden dimension (must be even)
    Returns:
        encoding: (B, K, dim) — positional encoding vectors
    """
    inv_freq = 1.0 / (10000 ** (
        torch.arange(0, dim, 2, device=positions.device, dtype=torch.float32) / dim
    ))  # (dim/2,)
    # (B, K, 1) x (1, 1, dim/2) -> (B, K, dim/2)
    freqs = positions.unsqueeze(-1) * inv_freq.unsqueeze(0).unsqueeze(0)
    return torch.cat([freqs.sin(), freqs.cos()], dim=-1)  # (B, K, dim)


class TransformerBlock(nn.Module):
    def __init__(self, hidden_size, n_heads, rope=None):
        super().__init__()
        self.norm1 = RMSNorm(hidden_size)
        self.attn = SelfAttention(hidden_size, n_heads, rope)
        self.norm2 = RMSNorm(hidden_size)
        self.ffn = FFN(hidden_size)
    
    def forward(self, x, causal_mask=None):
        x = x + self.attn(self.norm1(x), causal_mask)
        x = x + self.ffn(self.norm2(x))
        return x


# ============================================================
# RIBOSOME LAYER
# ============================================================

class RibosomeLayer(nn.Module):
    """
    The core innovation: scores tokens, detects boundaries,
    compresses into metatokens.
    
    Boundary detection uses Gumbel-softmax for differentiable
    discrete boundary placement during training.
    """
    
    def __init__(self, hidden_size, max_chunks=64, n_heads=4, temperature=1.0):
        super().__init__()
        self.hidden_size = hidden_size
        self.max_chunks = max_chunks
        
        # Importance scorer
        self.scorer = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, 1),
            nn.Sigmoid()
        )
        
        # Boundary predictor: outputs probability of boundary at each position
        self.boundary_predictor = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, 1),
        )
        
        # Chunk encoder: learnable query per chunk slot
        self.chunk_query = nn.Parameter(torch.randn(1, max_chunks, hidden_size) * 0.02)
        self.chunk_cross_attn = nn.MultiheadAttention(
            hidden_size, num_heads=n_heads, batch_first=True
        )
        self.chunk_norm = RMSNorm(hidden_size)
        
        # Temperature for Gumbel-softmax (annealed during training)
        self.gumbel_temperature = temperature
    
    def forward(self, hidden_states, return_debug=False, padding_mask=None):
        """
        Args:
            hidden_states: (B, S, H) from lower transformer
            padding_mask: (B, S) optional, True for real tokens, False for padding.
                         If None, all tokens are treated as real.
        Returns:
            chunk_repr: (B, K, H) compressed metatoken representations
            chunk_weights: (B, K) importance weight per chunk
            token_to_chunk: (B, S, K) soft assignment matrix
            importance: (B, S) per-token importance scores
        """
        B, S, H = hidden_states.shape
        K = min(self.max_chunks, S // 2)  # at most S/2 chunks
        
        # --- Score importance ---
        importance = self.scorer(hidden_states).squeeze(-1)  # (B, S)
        
        # Mask padding: zero importance and boundaries for padding tokens
        if padding_mask is not None:
            importance = importance * padding_mask.float()
        
        # --- Predict boundaries via Gumbel-softmax ---
        boundary_logits = self.boundary_predictor(hidden_states).squeeze(-1)  # (B, S)
        
        if self.training:
            # Gumbel-softmax: sample K boundary positions differentiably
            # Add Gumbel noise and take top-K via softmax
            gumbel_noise = -torch.log(-torch.log(
                torch.rand_like(boundary_logits) + 1e-8) + 1e-8)
            noisy_logits = (boundary_logits + gumbel_noise) / self.gumbel_temperature
        else:
            noisy_logits = boundary_logits
        
        # Soft boundary scores per position
        boundary_probs = torch.sigmoid(noisy_logits)  # (B, S)
        
        # Mask padding: zero boundary probs so padding doesn't create chunks
        if padding_mask is not None:
            boundary_probs = boundary_probs * padding_mask.float()
        
        # --- Build soft token-to-chunk assignment ---
        # Each token is assigned to the chunk defined by the nearest
        # boundary to its left. We use cumulative sum of boundary probs
        # as a soft "chunk index" and then compute assignment via
        # distance to each chunk slot.
        
        # Cumulative boundary creates a soft chunk index per token
        cum_boundaries = torch.cumsum(boundary_probs, dim=-1)  # (B, S)
        # Normalize to [0, K-1]
        max_cum = cum_boundaries[:, -1:].clamp(min=1.0)
        chunk_indices = cum_boundaries / max_cum * (K - 1)  # (B, S), values in [0, K-1]
        
        # Soft assignment: Gaussian kernel between token's chunk_index and each slot
        slots = torch.arange(K, device=hidden_states.device, dtype=torch.float32)
        slots = slots.unsqueeze(0).unsqueeze(0)  # (1, 1, K)
        chunk_indices_exp = chunk_indices.unsqueeze(-1)  # (B, S, 1)
        
        # Assignment weight: exp(-0.5 * (token_idx - slot)^2 / sigma^2)
        sigma = max(1.0, K / S * 2.0)  # adaptive width
        assign = torch.exp(-0.5 * ((chunk_indices_exp - slots) / sigma) ** 2)
        assign = assign / (assign.sum(dim=-1, keepdim=True) + 1e-8)  # normalize per token
        # assign: (B, S, K) — soft token-to-chunk assignment
        
        # Mask padding: zero assignment so padding tokens don't pollute chunks
        if padding_mask is not None:
            assign = assign * padding_mask.float().unsqueeze(-1)
        
        # --- Compress tokens into chunks ---
        # Weight assignment by importance (high-importance tokens dominate chunk repr)
        weighted_assign = assign * importance.unsqueeze(-1)  # (B, S, K)
        weighted_assign = weighted_assign / (weighted_assign.sum(dim=1, keepdim=True) + 1e-8)
        
        # Weighted sum of hidden states per chunk
        chunk_repr = torch.bmm(weighted_assign.transpose(1, 2), hidden_states)  # (B, K, H)
        
        # Refine via cross-attention with learnable queries
        queries = self.chunk_query[:, :K, :].expand(B, -1, -1)
        refined, _ = self.chunk_cross_attn(queries, chunk_repr, chunk_repr)
        chunk_repr = self.chunk_norm(chunk_repr + refined)
        
        # --- Compute chunk weights (for cascade priority) ---
        # Each chunk's weight = sum of importance scores it captures
        chunk_weights = torch.bmm(
            importance.unsqueeze(1),  # (B, 1, S)
            assign  # (B, S, K)
        ).squeeze(1)  # (B, K)
        
        # --- Compute chunk positions (weighted mean of source token positions) ---
        positions = torch.arange(S, device=hidden_states.device, dtype=torch.float32)
        # assign: (B, S, K) -> transpose to (B, K, S), multiply by positions (S,)
        chunk_positions = torch.bmm(
            assign.transpose(1, 2),  # (B, K, S)
            positions.view(1, S, 1).expand(B, -1, -1)  # (B, S, 1)
        ).squeeze(-1)  # (B, K)
        # Normalize by total assignment mass per chunk
        chunk_mass = assign.sum(dim=1)  # (B, K)
        chunk_positions = chunk_positions / (chunk_mass + 1e-8)  # (B, K)
        
        if return_debug:
            return chunk_repr, chunk_weights, assign, importance, boundary_probs, chunk_positions
        return chunk_repr, chunk_weights, assign, importance, chunk_positions


# ============================================================
# CASCADE PROCESSOR
# ============================================================

class CascadeProcessor(nn.Module):
    """
    Processes chunks in priority order (heaviest first).
    Uses causal attention so each chunk only sees already-processed
    heavier chunks. The anchor (heaviest) establishes context;
    lighter chunks are interpreted through the anchor's lens.
    """
    
    def __init__(self, hidden_size, n_heads, n_layers=2):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerBlock(hidden_size, n_heads)
            for _ in range(n_layers)
        ])
    
    def forward(self, chunk_repr, chunk_weights):
        """
        Args:
            chunk_repr: (B, K, H)
            chunk_weights: (B, K)
        Returns:
            processed: (B, K, H) — chunks processed in priority order
        """
        B, K, H = chunk_repr.shape
        
        # Sort by weight descending (heaviest first)
        sort_idx = chunk_weights.argsort(dim=-1, descending=True)
        sorted_repr = torch.gather(
            chunk_repr, 1, sort_idx.unsqueeze(-1).expand(-1, -1, H)
        )
        
        # Causal mask: chunk i can only attend to chunks 0..i
        causal = torch.triu(
            torch.ones(K, K, device=chunk_repr.device, dtype=torch.bool),
            diagonal=1
        )
        
        # Process through cascade layers
        x = sorted_repr
        for layer in self.layers:
            x = layer(x, causal_mask=causal)
        
        # Unsort back to original order
        unsort_idx = sort_idx.argsort(dim=-1)
        processed = torch.gather(
            x, 1, unsort_idx.unsqueeze(-1).expand(-1, -1, H)
        )
        
        return processed


# ============================================================
# CHUNK DECODER
# ============================================================

class ChunkDecoder(nn.Module):
    """
    Expands processed chunk representations back to token-level
    predictions. Each token cross-attends to the processed chunks,
    weighted by its original assignment.
    """
    
    def __init__(self, hidden_size, n_heads):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            hidden_size, num_heads=n_heads, batch_first=True
        )
        self.norm = RMSNorm(hidden_size)
        self.ffn = FFN(hidden_size)
        self.ffn_norm = RMSNorm(hidden_size)
    
    def forward(self, token_states, chunk_repr, token_to_chunk):
        """
        Args:
            token_states: (B, S, H) — original token representations from lower transformer
            chunk_repr: (B, K, H) — processed chunk representations
            token_to_chunk: (B, S, K) — soft assignment matrix
        Returns:
            expanded: (B, S, H) — token-level representations
        """
        # Cross-attention: tokens query, chunks are keys/values
        decoded, _ = self.cross_attn(token_states, chunk_repr, chunk_repr)
        x = self.norm(token_states + decoded)
        x = x + self.ffn(self.ffn_norm(x))
        return x


class ReverseRibosome(nn.Module):
    """
    The reverse ribosome: expands chunk-level representations back to
    full token-level resolution with enough capacity to recover
    fine-grained sequential detail.
    
    Pipeline:
      1. Cross-attention: each token attends to processed chunks
         (injects chunk-level context into token representations)
      2. Residual add with original token_states
         (preserves fine-grained token identity from embed layers)
      3. N causal self-attention layers at full token resolution
         (recovers sequential dependencies lost in compression)
    
    The self-attention layers are the key: they operate at full S
    tokens with causal masking, so they can reconstruct the
    fine-grained token-by-token predictions that compression destroyed.
    """
    
    def __init__(self, hidden_size, n_heads, n_layers=2, rope=None):
        super().__init__()
        # Step 1: cross-attention from tokens to chunks
        self.cross_attn = nn.MultiheadAttention(
            hidden_size, num_heads=n_heads, batch_first=True
        )
        self.cross_norm = RMSNorm(hidden_size)
        self.cross_ffn = FFN(hidden_size)
        self.cross_ffn_norm = RMSNorm(hidden_size)
        
        # Step 2: token-level self-attention layers (with RoPE for position)
        self.layers = nn.ModuleList([
            TransformerBlock(hidden_size, n_heads, rope)
            for _ in range(n_layers)
        ])
        self.out_norm = RMSNorm(hidden_size)
    
    def forward(self, token_states, chunk_repr, token_to_chunk):
        """
        Args:
            token_states: (B, S, H) — from embed layers (has positional info)
            chunk_repr: (B, K, H) — processed chunk representations
            token_to_chunk: (B, S, K) — soft assignment matrix
        Returns:
            output: (B, S, H) — refined token-level representations
        """
        B, S, H = token_states.shape
        
        # Cross-attention: inject chunk context into token representations
        decoded, _ = self.cross_attn(token_states, chunk_repr, chunk_repr)
        x = self.cross_norm(token_states + decoded)
        x = x + self.cross_ffn(self.cross_ffn_norm(x))
        
        # Causal self-attention at full token resolution
        causal = torch.triu(
            torch.ones(S, S, device=x.device, dtype=torch.bool), diagonal=1
        )
        for layer in self.layers:
            x = layer(x, causal_mask=causal)
        
        return self.out_norm(x)


# ============================================================
# FULL MODEL
# ============================================================

class RibosomeCascadeNative(nn.Module):
    """
    Full native Ribosome-Cascade transformer.
    
    Architecture:
        Embedding → Lower Transformer (4L) → Ribosome Layer →
        Cascade Processor (2L) → Upper Transformer (4L) →
        Chunk Decoder → LM Head
    """
    
    def __init__(
        self,
        vocab_size=50257,
        hidden_size=768,
        n_heads=12,
        lower_layers=4,
        upper_layers=4,
        cascade_layers=2,
        max_seq_len=1024,
        max_chunks=64,
        dropout=0.0,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.max_seq_len = max_seq_len
        
        # Token embedding (weight-tied with LM head)
        self.tok_emb = nn.Embedding(vocab_size, hidden_size)
        
        # Rotary embeddings
        self.rope = RotaryEmbedding(hidden_size // n_heads, max_seq_len)
        
        # Lower transformer: token-level processing
        self.lower = nn.ModuleList([
            TransformerBlock(hidden_size, n_heads, self.rope)
            for _ in range(lower_layers)
        ])
        self.lower_norm = RMSNorm(hidden_size)
        
        # Ribosome layer: score + group + compress
        self.ribosome = RibosomeLayer(hidden_size, max_chunks, n_heads)
        
        # Cascade processor: priority-ordered chunk attention
        self.cascade = CascadeProcessor(hidden_size, n_heads, cascade_layers)
        
        # Upper transformer: chunk-level processing
        self.upper = nn.ModuleList([
            TransformerBlock(hidden_size, n_heads)
            for _ in range(upper_layers)
        ])
        self.upper_norm = RMSNorm(hidden_size)
        
        # Chunk decoder: expand back to tokens
        self.decoder = ChunkDecoder(hidden_size, n_heads)
        
        # LM head (tied weights with embedding)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # weight tying
        
        # Training control
        self.ribosome_alpha = 1.0  # 0→1 ramp during training
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                torch.nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None:
                    torch.nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                torch.nn.init.normal_(m.weight, std=0.02)
    
    def forward(self, input_ids, labels=None):
        B, S = input_ids.shape
        
        # Embed
        x = self.tok_emb(input_ids)  # (B, S, H)
        
        # Causal mask for token-level attention
        causal = torch.triu(
            torch.ones(S, S, device=x.device, dtype=torch.bool),
            diagonal=1
        )
        
        # Lower transformer
        for layer in self.lower:
            x = layer(x, causal_mask=causal)
        token_states = self.lower_norm(x)  # save for decoder
        
        # Ribosome: compress to chunks
        chunk_repr, chunk_weights, assign, importance, chunk_positions = self.ribosome(token_states)
        
        # Inject sequence position metadata into chunks
        chunk_repr = chunk_repr + sinusoidal_position_encoding(
            chunk_positions, self.hidden_size)
        
        # Cascade: priority-ordered processing
        chunk_repr = self.cascade(chunk_repr, chunk_weights)
        
        # Upper transformer: chunk-level
        for layer in self.upper:
            chunk_repr = layer(chunk_repr)
        chunk_repr = self.upper_norm(chunk_repr)
        
        # Decode: expand chunks back to token-level
        decoded = self.decoder(token_states, chunk_repr, assign)
        
        # Blend with bypass (alpha ramp for training stability)
        output = self.ribosome_alpha * decoded + (1 - self.ribosome_alpha) * token_states
        
        # LM head
        logits = self.lm_head(output)
        
        loss = None
        if labels is not None:
            # No shift: loader already provides aligned input/label pairs
            ce_loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                labels.view(-1)
            )
            # Sparsity regularization: very gentle, let model learn importance first
            sparsity_loss = importance.mean()
            # Boundary regularization: encourage ~S/4 boundaries
            target_boundaries = S / 4
            boundary_count = assign.sum(dim=1).max(dim=-1).values.mean()
            boundary_loss = (boundary_count - target_boundaries).pow(2) / target_boundaries
            
            loss = ce_loss + 0.001 * sparsity_loss + 0.0001 * boundary_loss
        
        return loss, logits, importance
    
    def count_params(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total, trainable


# ============================================================
# QUICK VALIDATION
# ============================================================

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model = RibosomeCascadeNative(
        vocab_size=50257,
        hidden_size=768,
        n_heads=12,
        lower_layers=4,
        upper_layers=4,
        cascade_layers=2,
        max_seq_len=1024,
        max_chunks=64,
    ).to(device)
    
    total, trainable = model.count_params()
    print(f"Total params:     {total:,}")
    print(f"Trainable params: {trainable:,}")
    
    # Smoke test
    x = torch.randint(0, 50257, (2, 128)).to(device)
    labels = x.clone()
    
    loss, logits, importance = model(x, labels)
    print(f"Loss: {loss.item():.4f}")
    print(f"Logits shape: {logits.shape}")
    print(f"Importance shape: {importance.shape}")
    print(f"Importance range: [{importance.min().item():.4f}, {importance.max().item():.4f}]")
    
    loss.backward()
    print("Backward pass OK")
    print(f"GPU memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")


In [ ]:
assert 'CONFIG' in globals(), "Run Cell 1 (Setup) first - it defines CONFIG."
import sys
sys.path.insert(0, '/content')
import torch, torch.nn as nn, torch.nn.functional as F
from native_arch_v1 import (
    RMSNorm, RotaryEmbedding, apply_rotary, SelfAttention, FFN,
    TransformerBlock, RibosomeLayer, ReverseRibosome, ChunkDecoder,
)

class RibosomeTiny(nn.Module):
    """
    Ribosome compresses 256→16 tokens, then a tiny 4-layer transformer
    processes metatokens. Much smaller total compute than the big baseline.
    
    With reverse_layers > 0, uses ReverseRibosome for token-level
    reconstruction with causal self-attention after chunk decompression.
    """
    def __init__(self, vocab_size=50257, hidden_size=512, n_heads=8,
                 embed_layers=2, upper_layers=4, max_seq_len=256, n_chunks=16,
                 reverse_layers=0):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, hidden_size)
        self.rope = RotaryEmbedding(hidden_size // n_heads, max_seq_len)

        # Light embedding layers (build token representations)
        self.embed = nn.ModuleList([
            TransformerBlock(hidden_size, n_heads, self.rope)
            for _ in range(embed_layers)
        ])
        self.embed_norm = RMSNorm(hidden_size)

        # Ribosome: compress
        self.ribosome = RibosomeLayer(hidden_size, n_chunks, n_heads)

        # Tiny upper transformer on metatokens
        self.upper = nn.ModuleList([
            TransformerBlock(hidden_size, n_heads)
            for _ in range(upper_layers)
        ])
        self.upper_norm = RMSNorm(hidden_size)

        # Decode back to tokens
        if reverse_layers > 0:
            self.decoder = ReverseRibosome(hidden_size, n_heads,
                                           n_layers=reverse_layers, rope=self.rope)
        else:
            self.decoder = ChunkDecoder(hidden_size, n_heads)

        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

        self.ribosome_alpha = 1.0
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                torch.nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None:
                    torch.nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                torch.nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, labels=None, padding_mask=None):
        B, S = input_ids.shape
        
        # Auto-derive padding mask from labels if not provided
        if padding_mask is None and labels is not None:
            if (labels == -100).any():
                padding_mask = (labels != -100)
        
        x = self.tok_emb(input_ids)
        causal = torch.triu(
            torch.ones(S, S, device=x.device, dtype=torch.bool), diagonal=1)

        for layer in self.embed:
            x = layer(x, causal_mask=causal)
        token_states = self.embed_norm(x)

        chunk_repr, chunk_weights, assign, importance, chunk_positions = self.ribosome(
            token_states, padding_mask=padding_mask)

        # Note: chunk_positions available for future use but NOT injected
        # as additive sinusoidal encoding (that hurt PPL). Instead, the
        # ReverseRibosome uses RoPE at token-level on the way out.

        for layer in self.upper:
            chunk_repr = layer(chunk_repr)
        chunk_repr = self.upper_norm(chunk_repr)

        decoded = self.decoder(token_states, chunk_repr, assign)
        output = self.ribosome_alpha * decoded + (1 - self.ribosome_alpha) * token_states

        logits = self.lm_head(output)
        loss = None
        if labels is not None:
            # Data loader already shifts: input=tokens[:-1], labels=tokens[1:]
            # So logits[i] predicts position i+1 and labels[i] IS position i+1
            # No extra shift needed.
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                labels.view(-1))
        return loss, logits, importance

    def count_params(self):
        return sum(p.numel() for p in self.parameters())


# Instantiate and size-check
torch.manual_seed(CONFIG['seed'])
model = RibosomeTiny(
    vocab_size=CONFIG['vocab_size'], hidden_size=CONFIG['hidden_size'],
    n_heads=CONFIG['n_heads'], embed_layers=CONFIG['embed_layers'],
    upper_layers=CONFIG['upper_layers'], max_seq_len=CONFIG['max_seq_len'],
    n_chunks=CONFIG['n_chunks'], reverse_layers=CONFIG['reverse_layers'],
)
n_params = model.count_params()
print(f"Parameters: {n_params/1e6:.2f}M")
print(f"Estimated activation memory at bs={CONFIG['batch_size']}, seq={CONFIG['max_seq_len']}, bf16: "
      f"~{CONFIG['batch_size']*CONFIG['max_seq_len']*CONFIG['hidden_size']*2*16/1e9:.2f} GB (rough)")


In [ ]:
assert 'CONFIG' in globals(), "Run Cell 1 (Setup) first - it defines CONFIG."
from transformers import GPT2TokenizerFast
from datasets import load_dataset

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SEQ = CONFIG['max_seq_len']
BS  = CONFIG['batch_size']

def c4_stream_iter(split="train", buffer_tokens=200_000):
    """Streaming C4 -> fixed-length packed sequences of length SEQ+1.
    Yields (input_ids[:SEQ], labels[1:SEQ+1]) pairs batched to BS."""
    ds = load_dataset("allenai/c4", CONFIG['train_subset'], split=split, streaming=True)
    eot = tokenizer.eos_token_id
    buf = []
    for ex in ds:
        ids = tokenizer(ex["text"], truncation=False, add_special_tokens=False)["input_ids"]
        buf.extend(ids); buf.append(eot)
        while len(buf) >= (SEQ + 1) * BS:
            chunk = buf[:(SEQ + 1) * BS]
            buf   = buf[(SEQ + 1) * BS:]
            t = torch.tensor(chunk, dtype=torch.long).view(BS, SEQ + 1)
            yield t[:, :SEQ].contiguous(), t[:, 1:].contiguous()

def wt103_val_batches(max_batches=None):
    ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="validation")
    txt = "\n".join(ds["text"])
    ids = tokenizer(txt, truncation=False, add_special_tokens=False)["input_ids"]
    n = (len(ids) - 1) // (SEQ * BS)
    if max_batches is not None: n = min(n, max_batches)
    for b in range(n):
        chunk = ids[b*SEQ*BS : (b+1)*SEQ*BS + 1]
        t = torch.tensor(chunk[:-1], dtype=torch.long).view(BS, SEQ)
        l = torch.tensor(chunk[1:],  dtype=torch.long).view(BS, SEQ)
        yield t, l

print("Tokenizer:", tokenizer, "vocab:", tokenizer.vocab_size)


In [ ]:
# LAMBADA: score the last word given the full context. Accuracy = top-1 match.
def build_lambada_examples(n_examples=None):
    ds = load_dataset("EleutherAI/lambada_openai", "en", split="test")
    out = []
    for ex in ds:
        text = ex["text"].strip()
        if " " not in text: continue
        ctx, last = text.rsplit(" ", 1)
        ctx_ids  = tokenizer(" " + ctx,  add_special_tokens=False)["input_ids"]
        last_ids = tokenizer(" " + last, add_special_tokens=False)["input_ids"]
        if len(last_ids) == 0: continue
        # Truncate context to leave room for last_ids within SEQ.
        max_ctx = SEQ - len(last_ids) - 1
        if max_ctx <= 0:
            last_ids = last_ids[:SEQ - 2]; max_ctx = SEQ - len(last_ids) - 1
        ctx_ids = ctx_ids[-max_ctx:] if len(ctx_ids) > max_ctx else ctx_ids
        out.append((ctx_ids, last_ids))
        if n_examples is not None and len(out) >= n_examples: break
    return out

@torch.no_grad()
def eval_ce(model, batches, device, max_batches=None, dtype=torch.bfloat16):
    model.eval()
    tot, tok = 0.0, 0
    for i, (x, y) in enumerate(batches):
        if max_batches is not None and i >= max_batches: break
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast('cuda', dtype=dtype):
            loss, _, _ = model(x, labels=y)
        ntok = y.numel()
        tot += loss.item() * ntok; tok += ntok
    avg = tot / max(tok, 1)
    return {"ce": avg, "ppl": math.exp(avg), "tokens": tok}

@torch.no_grad()
def eval_lambada(model, examples, device, dtype=torch.bfloat16):
    model.eval()
    correct, total, total_last_tok_ce, n_last_tok = 0, 0, 0.0, 0
    for ctx_ids, last_ids in examples:
        full = ctx_ids + last_ids
        if len(full) < 2: continue
        x = torch.tensor(full[:-1], dtype=torch.long, device=device).unsqueeze(0)
        y = torch.tensor(full[1:],  dtype=torch.long, device=device).unsqueeze(0)
        # pad if shorter than SEQ
        if x.size(1) < SEQ:
            pad = SEQ - x.size(1)
            x = F.pad(x, (0, pad), value=tokenizer.eos_token_id)
            y = F.pad(y, (0, pad), value=-100)
        with torch.amp.autocast('cuda', dtype=dtype):
            _, logits, _ = model(x, labels=y)
        L = len(full) - 1
        start = L - len(last_ids)
        pred = logits[0, start:L].argmax(dim=-1).tolist()
        targ = last_ids
        if pred == targ: correct += 1
        # last-token CE for PPL-on-last-word
        valid = logits[0, start:L]
        ce = F.cross_entropy(valid, torch.tensor(targ, device=device), reduction='sum').item()
        total_last_tok_ce += ce; n_last_tok += len(targ)
        total += 1
    return {"acc": correct/max(total,1), "n": total,
            "last_word_ppl": math.exp(total_last_tok_ce / max(n_last_tok,1))}


In [ ]:
def get_lr(step):
    w, tot = CONFIG['warmup_steps'], CONFIG['total_steps']
    mx, mn = CONFIG['max_lr'], CONFIG['min_lr']
    if step < w:
        return mx * step / max(w, 1)
    t = (step - w) / max(tot - w, 1)
    t = min(max(t, 0.0), 1.0)
    return mn + 0.5 * (mx - mn) * (1 + math.cos(math.pi * t))

def get_alpha(step):
    r = CONFIG['alpha_ramp_steps']
    return min(1.0, step / max(r, 1))

def save_ckpt(path, model, opt, step, best_ppl):
    torch.save({
        "step": step, "model": model.state_dict(),
        "opt": opt.state_dict(), "best_ppl": best_ppl,
        "config": CONFIG,
    }, path)

def load_ckpt(path, model, opt, device):
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck["model"])
    opt.load_state_dict(ck["opt"])
    return ck["step"], ck.get("best_ppl", float("inf"))


In [ ]:
device = torch.device("cuda")
dtype = torch.bfloat16 if CONFIG['dtype'] == "bf16" else torch.float16

torch.manual_seed(CONFIG['seed']); random.seed(CONFIG['seed'])

model = model.to(device)
opt = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['max_lr'], weight_decay=CONFIG['weight_decay'],
    betas=(CONFIG['beta1'], CONFIG['beta2']), eps=1e-8, fused=True,
)

# Resume if checkpoint exists on Drive.
start_step, best_ppl = 0, float("inf")
if os.path.exists(CKPT_PATH):
    start_step, best_ppl = load_ckpt(CKPT_PATH, model, opt, device)
    print(f"Resumed from step {start_step}, best PPL {best_ppl:.3f}")
else:
    print("Fresh start.")

# Pre-build LAMBADA mini-set for mid-training gut checks (500 examples ~ 1min)
lambada_mini = build_lambada_examples(n_examples=500)
print(f"LAMBADA mini set: {len(lambada_mini)} examples")

log_f = open(LOG_PATH, "a")
def log(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line); log_f.write(line + "\n"); log_f.flush()

log(f"=== Training start === step={start_step} total={CONFIG['total_steps']}")

train_iter = c4_stream_iter()
step = start_step
model.train()
t0 = time.time(); n_tok = 0; loss_ema = None

try:
    while step < CONFIG['total_steps']:
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = c4_stream_iter(); x, y = next(train_iter)
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        lr = get_lr(step)
        for g in opt.param_groups: g["lr"] = lr
        model.ribosome_alpha = get_alpha(step)

        with torch.amp.autocast('cuda', dtype=dtype):
            loss, _, _ = model(x, labels=y)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        opt.step()

        loss_ema = loss.item() if loss_ema is None else 0.98 * loss_ema + 0.02 * loss.item()
        n_tok += x.numel(); step += 1

        if step % CONFIG['log_every'] == 0:
            tps = n_tok / max(time.time() - t0, 1e-6)
            log(f"step {step:6d} | loss {loss.item():.4f} (ema {loss_ema:.4f}) | "
                f"lr {lr:.2e} | alpha {model.ribosome_alpha:.3f} | {tps/1000:.1f}K tok/s")

        if step % CONFIG['eval_every'] == 0 or step == CONFIG['total_steps']:
            r_wt = eval_ce(model, wt103_val_batches(max_batches=CONFIG['mid_eval_wt103_batches']),
                           device, dtype=dtype)
            r_lb = eval_lambada(model, lambada_mini, device, dtype=dtype)
            log(f"  >> mid-eval: wt103 PPL {r_wt['ppl']:.3f} | "
                f"LAMBADA mini acc {r_lb['acc']*100:.2f}% ({r_lb['n']} ex) | "
                f"last-word PPL {r_lb['last_word_ppl']:.3f}")
            if r_wt["ppl"] < best_ppl:
                best_ppl = r_wt["ppl"]
                save_ckpt(BEST_PATH, model, opt, step, best_ppl)
                log(f"  >> new best {best_ppl:.3f}, saved {BEST_PATH}")
            model.train()

        if step % CONFIG['ckpt_every'] == 0:
            save_ckpt(CKPT_PATH, model, opt, step, best_ppl)
finally:
    save_ckpt(CKPT_PATH, model, opt, step, best_ppl)
    log("=== Training loop exited ===")
    log_f.close()


In [ ]:
# Reload best checkpoint for final evaluation.
if os.path.exists(BEST_PATH):
    ck = torch.load(BEST_PATH, map_location=device)
    model.load_state_dict(ck["model"])
    print(f"Loaded best ckpt from step {ck['step']}, best wt103 PPL {ck['best_ppl']:.3f}")

model.eval()

# Full wt103 val
wt_full = eval_ce(model, wt103_val_batches(), device, dtype=dtype)
print(f"wt103 (full val): CE {wt_full['ce']:.4f} | PPL {wt_full['ppl']:.3f} | tokens {wt_full['tokens']}")

# C4 held-out
c4_val = eval_ce(model, c4_stream_iter(split="validation"), device,
                 max_batches=CONFIG['mid_eval_c4_batches'], dtype=dtype)
print(f"C4 (500-batch val): CE {c4_val['ce']:.4f} | PPL {c4_val['ppl']:.3f}")

# Full LAMBADA (openai split, ~5153 examples)
lambada_full = build_lambada_examples()
print(f"LAMBADA full set: {len(lambada_full)} examples; evaluating...")
lb_full = eval_lambada(model, lambada_full, device, dtype=dtype)
print(f"LAMBADA: acc {lb_full['acc']*100:.2f}% | last-word PPL {lb_full['last_word_ppl']:.3f} | n {lb_full['n']}")

# Optional HellaSwag - best-effort (skip on any failure, too noisy for an undertrained model anyway)
hs = None
try:
    hs_ds = load_dataset("hellaswag", split="validation")
    correct, total = 0, 0
    with torch.no_grad():
        for ex in hs_ds.select(range(min(1000, len(hs_ds)))):
            ctx = ex["ctx"]; endings = ex["endings"]; label = int(ex["label"])
            scores = []
            for e in endings:
                full = tokenizer(ctx + " " + e, truncation=True, max_length=SEQ,
                                 add_special_tokens=False)["input_ids"]
                if len(full) < 2: scores.append(float("inf")); continue
                x = torch.tensor(full[:-1], dtype=torch.long, device=device).unsqueeze(0)
                y = torch.tensor(full[1:],  dtype=torch.long, device=device).unsqueeze(0)
                if x.size(1) < SEQ:
                    pad = SEQ - x.size(1)
                    x = F.pad(x, (0, pad), value=tokenizer.eos_token_id)
                    y = F.pad(y, (0, pad), value=-100)
                with torch.amp.autocast('cuda', dtype=dtype):
                    loss, _, _ = model(x, labels=y)
                scores.append(loss.item())
            pred = int(torch.tensor(scores).argmin().item())
            if pred == label: correct += 1
            total += 1
    hs = {"acc": correct/max(total,1), "n": total}
    print(f"HellaSwag (1K subset): acc {hs['acc']*100:.2f}% | n {hs['n']}")
except Exception as e:
    print(f"HellaSwag skipped: {e}")

results = {
    "run_tag": RUN_TAG, "config": CONFIG,
    "params_M": n_params / 1e6,
    "wt103_full": wt_full, "c4_val_500batch": c4_val,
    "lambada_full": lb_full, "hellaswag_1k": hs,
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
with open(RESULTS, "w") as f:
    json.dump(results, f, indent=2)
print(f"Results written to {RESULTS}")


## Interpreting results

| LAMBADA acc | Interpretation | Next step |
|-------------|----------------|-----------|
| > 10%       | Architecture works at scale. The 0-0.3% dead zone was undertraining + too-high compression. | Scale further (hidden=1024, 300K steps, chunks=32/64 Pareto). Write up. |
| 1-10%       | Marginal. Need matched-FLOPs baseline comparison. | Run a 63M-param dense baseline on the same C4 budget, compare PPL + LAMBADA. |
| < 1%        | Hourglass as specified cannot recover last-word semantics, even at ~GPT-2-Small scale. | Reframe project: characterize Pareto frontier of PPL/bit vs compression ratio. LAMBADA is not the right benchmark. |

## What to check even before final eval:
1. wt103 mid-training PPL should be descending monotonically past ~20 by step 40K.
2. LAMBADA mini acc should move off exact-0% by step 20K if it's ever going to.
3. Gumbel `tau` is fixed at 1.0 throughout; if boundaries look random in `assign`, anneal.
4. If memory is tight: drop `batch_size` to 32 and `grad_accum=2` - keeps token-per-step constant.

## Post-mortem checklist (fill in after run):
- [ ] Final wt103 PPL vs best prior run (RibosomeTiny 256-chunks sweep: best was ~2.3 at chunks=64 but LAMBADA 0%)
- [ ] LAMBADA acc vs prior (all prior runs: 0.00% - 0.33%)
- [ ] Loss curve shape - does it plateau, or still descending at step 100K?
- [ ] Compare wallclock efficiency to local 3060 Ti baseline
